In [4]:
from sklearnex import patch_sklearn
patch_sklearn()

Extension for scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [5]:
import polars as pl 

In [6]:
df = pl.read_csv('insurance.csv')
df

age,sex,bmi,children,smoker,region,charges
i64,str,f64,i64,str,str,f64
19,"""female""",27.9,0,"""yes""","""southwest""",16884.924
18,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28,"""male""",33.0,3,"""no""","""southeast""",4449.462
33,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32,"""male""",28.88,0,"""no""","""northwest""",3866.8552
…,…,…,…,…,…,…
50,"""male""",30.97,3,"""no""","""northwest""",10600.5483
18,"""female""",31.92,0,"""no""","""northeast""",2205.9808
18,"""female""",36.85,0,"""no""","""southeast""",1629.8335


In [7]:
df.head()

age,sex,bmi,children,smoker,region,charges
i64,str,f64,i64,str,str,f64
19,"""female""",27.9,0,"""yes""","""southwest""",16884.924
18,"""male""",33.77,1,"""no""","""southeast""",1725.5523
28,"""male""",33.0,3,"""no""","""southeast""",4449.462
33,"""male""",22.705,0,"""no""","""northwest""",21984.47061
32,"""male""",28.88,0,"""no""","""northwest""",3866.8552


In [8]:
# Pisahkan fitur (X) dan target (y).
X = df.drop('charges')
y = df['charges']

In [9]:
import polars.selectors as pl_selectors

In [10]:
# Identifikasi kolom numerik dan kategorikal.
X_numeric = X.select(pl_selectors.numeric()).columns
X_categorical = X.select(~pl_selectors.numeric()).columns
X_numeric, X_categorical

(['age', 'bmi', 'children'], ['sex', 'smoker', 'region'])

In [11]:
from sklearn.model_selection import train_test_split

# Bagi data menjadi train set dan test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


X_train_numeric = X_train[X_numeric]
X_train_categorical = X_train[X_categorical]

X_test_numeric = X_test[X_numeric]
X_test_categorical = X_test[X_categorical]

In [12]:
# Buat preprocessing untuk fitur numerik dan kategorikal.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), X_numeric),
        ('cat', OneHotEncoder(), X_categorical)
    ]
)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)


In [13]:
# Custom RFM parameters
rfm_params = {
    'model': {
        'kernel': 'l2',           # Kernel type
        'bandwidth': 5.0,         # Kernel bandwidth
        'exponent': 1.0,          # Kernel exponent
        'diag': False,            # Diagonal Mahalanobis matrix
        'bandwidth_mode': 'constant',  # Maximum leaf size for tree-based methods
    },
    'fit': {
        'reg': 1e-3,              # Regularization parameter
        'iters': 10,               # Number of iterations
        'M_batch_size': 500,     # Batch size for AGOP
        'verbose': True,          # Verbose output
        'early_stop_rfm': True    # Early stopping
    }
}


In [14]:
import numpy as np
import optuna
from sklearn.metrics import mean_squared_error
from xrfm import xRFM

leaf_candidates = [
    n for n in [256, 512, 1024, 2048, 4096, 8192, 16384]
    if n < len(X_train_preprocessed)
]

if not leaf_candidates:
    leaf_candidates = [max(2, len(X_train_preprocessed) // 2)]


def objective(trial):
    rfm_params = {
        "model": {
            "kernel": "l2_high_dim",
            "bandwidth": trial.suggest_float(
                "bandwidth", 0.5, 20.0, log=True
            ),
            "exponent": trial.suggest_float(
                "exponent", 0.5, 2.0
            ),
            "diag": True,
            "bandwidth_mode": "constant",
        },
        "fit": {
            "reg": trial.suggest_float(
                "reg", 1e-6, 1e-1, log=True
            ),
            "iters": trial.suggest_categorical(
                "iters", [2, 3, 5]
            ),
            "M_batch_size": 1000,
            "early_stop_rfm": True,
            "verbose": False,
        },
    }

    model = xRFM(
        rfm_params=rfm_params,
        device="cpu",
        tuning_metric="mse",
        max_leaf_size=trial.suggest_categorical(
            "max_leaf_size", leaf_candidates
        ),
        n_trees=trial.suggest_int("n_trees", 1, 3),
        split_method="top_vector_agop_on_subset",
        n_threads=4,
        random_state=42,
        verbose=False,
    )

    model.fit(
        X_train_preprocessed, y_train,
        X_test_preprocessed, y_test,
    )

    pred = np.asarray(
        model.predict(X_test_preprocessed)
    ).reshape(-1)

    return float(np.sqrt(
        mean_squared_error(y_test, pred)
    ))


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(objective, n_trials=50)

print("Best RMSE:", study.best_value)
print("Best parameters:", study.best_params)

/home/data/kuliah/rka/ML/Asset/Modul-Praktikum-ML-RKA/5. Supervised Learning Regression/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-10-26 08:54:23,044] A new study created in memory with name: no-name-444c1fdc-f63e-4de1-9a82-9727b310df1c
[I 2026-10-26 08:54:23,818] Trial 0 finished with value: 4863.981800077725 and parameters: {'bandwidth': 1.990699667393337, 'exponent': 1.9260714596148742, 'reg': 0.004570563099801453, 'iters': 2, 'max_leaf_size': 512, 'n_trees': 3}. Best is trial 0 with value: 4863.981800077725.
[I 2026-10-26 08:54:24,076] Trial 1 finished with value: 8326.258092406566 and parameters: {'bandwidth': 0.5394455304087532, 'exponent': 1.9548647782429915, 'reg': 0.014528246637516038, 'iters': 2, 'max_leaf_size': 512, 'n_trees': 1}. Best is trial 0 with value: 4863.981800077

Best RMSE: 4161.715280629394
Best parameters: {'bandwidth': 8.166172994656634, 'exponent': 1.987654424200511, 'reg': 0.0010044768195991852, 'iters': 5, 'max_leaf_size': 256, 'n_trees': 2}


In [15]:
import numpy as np
import torch
from xrfm import xRFM

p = study.best_params

rfm_params = {
    "model": {
        "kernel": "l2_high_dim",
        "bandwidth": p["bandwidth"],
        "exponent": p["exponent"],
        "diag": True,
        "bandwidth_mode": "constant",
    },
    "fit": {
        "reg": p["reg"],
        "iters": p["iters"],
        "M_batch_size": 1000,
        "early_stop_rfm": True,
        "verbose": False,
    },
}

best_model = xRFM(
    rfm_params=rfm_params,
    device=torch.device("cpu"),
    tuning_metric="mse",
    max_leaf_size=p["max_leaf_size"],
    n_trees=p["n_trees"],
    split_method="top_vector_agop_on_subset",
    n_threads=4,
    random_state=42,
    verbose=False,
)

best_model.fit(
    X_train_preprocessed, y_train,
    X_test_preprocessed, y_test,
)

In [16]:
from lightgbm import LGBMRegressor
from sklearn.svm import SVR

In [17]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

In [18]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

In [21]:
import numpy as np
import torch

from sklearn.base import clone
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor


# Factory supaya konfigurasi xRFM konsisten, tetapi instansinya selalu baru
def make_xrfm():
    return xRFM(
        rfm_params=rfm_params,
        device=torch.device("cpu"),
        tuning_metric="mse",
        max_leaf_size=p["max_leaf_size"],
        n_trees=p["n_trees"],
        split_method="top_vector_agop_on_subset",
        n_threads=4,
        random_state=42,
        verbose=False,
    )


# Base models
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    n_jobs=4,
)

poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression(),
)


# ==================================================
# 1. GENERATE OOF PREDICTIONS
# ==================================================

kfolds = 5
kf = KFold(n_splits=kfolds, shuffle=True, random_state=42)

oof_predictions = np.zeros(
    (len(X_train_preprocessed), 3)
)
y_train_np = y_train.to_numpy().ravel()
for fold, (train_index, val_index) in enumerate(
    kf.split(X_train_preprocessed)
):
    X_train_fold = X_train_preprocessed[train_index]
    X_val_fold = X_train_preprocessed[val_index]

    y_train_fold = y_train_np[train_index]
    y_val_fold = y_train_np[val_index]

    # xRFM membutuhkan validation internal.
    # Jangan gunakan outer validation untuk early stopping.
    X_fit, X_inner_val, y_fit, y_inner_val = train_test_split(
        X_train_fold,
        y_train_fold,
        test_size=0.15,
        random_state=100 + fold,
    )

    # xRFM: instansiasi baru pada setiap fold
    xrfm_fold = make_xrfm()
    xrfm_fold.fit(
        X_fit,
        y_fit,
        X_inner_val,
        y_inner_val,
    )

    # XGBoost: clone dan fit pada outer training fold
    xgb_fold = clone(xgb_model)
    xgb_fold.fit(X_train_fold, y_train_fold)

    # Polynomial Regression: clone dan fit pada outer training fold
    poly_fold = clone(poly_model)
    poly_fold.fit(X_train_fold, y_train_fold)

    # Prediksi hanya pada outer validation fold
    oof_predictions[val_index, 0] = np.asarray(
        xrfm_fold.predict(X_val_fold)
    ).reshape(-1)

    oof_predictions[val_index, 1] = xgb_fold.predict(
        X_val_fold
    ).reshape(-1)

    oof_predictions[val_index, 2] = poly_fold.predict(
        X_val_fold
    ).reshape(-1)


# ==================================================
# 2. TRAIN FINAL MODELS
# ==================================================

# Model scikit-learn di-fit ulang menggunakan seluruh training data
xgb_final = clone(xgb_model)
xgb_final.fit(X_train_preprocessed, y_train)

poly_final = clone(poly_model)
poly_final.fit(X_train_preprocessed, y_train)

# xRFM: validation internal diambil dari training data.
# X_test tidak boleh dipakai untuk validation atau tuning.
X_fit, X_inner_val, y_fit, y_inner_val = train_test_split(
    X_train_preprocessed,
    y_train,
    test_size=0.15,
    random_state=42,
)

xrfm_final = make_xrfm()
xrfm_final.fit(
    X_fit,
    y_fit,
    X_inner_val,
    y_inner_val,
)


# ==================================================
# 3. TEST PREDICTIONS
# ==================================================

pred_xrfm = np.asarray(
    xrfm_final.predict(X_test_preprocessed)
).reshape(-1)

pred_xgb = xgb_final.predict(X_test_preprocessed).reshape(-1)
pred_poly = poly_final.predict(X_test_preprocessed).reshape(-1)

pred_ensemble = (pred_xrfm + pred_xgb + pred_poly) / 3


# ==================================================
# 4. EVALUATION
# ==================================================

for name, pred in [
    ("xRFM", pred_xrfm),
    ("XGBoost", pred_xgb),
    ("Polynomial Regression", pred_poly),
    ("Ensemble", pred_ensemble),
]:
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"{name}: RMSE = {rmse:.6f}")
    print(f"{name}: R2   = {r2:.6f}")

xRFM: RMSE = 4332.512043
xRFM: R2   = 0.879093
XGBoost: RMSE = 4248.007124
XGBoost: R2   = 0.883764
Polynomial Regression: RMSE = 4551.132385
Polynomial Regression: R2   = 0.866583
Ensemble: RMSE = 4219.848282
Ensemble: R2   = 0.885299


In [20]:
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, r2_score

# Target training
y_train_np = np.asarray(y_train.to_numpy()).reshape(-1)
y_test_np = np.asarray(y_test.to_numpy()).reshape(-1)

# 1. Cari bobot ensemble optimal berdasarkan OOF predictions
def objective(weights):
    pred = oof_predictions @ weights
    return mean_squared_error(y_train_np, pred)

result = minimize(
    objective,
    x0=np.array([1/3, 1/3, 1/3]),
    method="SLSQP",
    bounds=[(0, 1)] * 3,
    constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1},
)

weights = result.x

print("Optimal weights:")
print(f"xRFM:   {weights[0]:.4f}")
print(f"XGBoost: {weights[1]:.4f}")
print(f"Poly:    {weights[2]:.4f}")

# 2. Gabungkan prediksi test menggunakan bobot optimal
test_predictions = np.column_stack([
    pred_xrfm,
    pred_xgb,
    pred_poly,
])

pred_ensemble = test_predictions @ weights

# 3. Evaluasi
for name, pred in [
    ("xRFM", pred_xrfm),
    ("XGBoost", pred_xgb),
    ("Polynomial Regression", pred_poly),
    ("Optimized Ensemble", pred_ensemble),
]:
    rmse = np.sqrt(mean_squared_error(y_test_np, pred))
    r2 = r2_score(y_test_np, pred)

    print(f"{name}: RMSE = {rmse:.6f}")
    print(f"{name}: R2   = {r2:.6f}")

Optimal weights:
xRFM:   0.3333
XGBoost: 0.3333
Poly:    0.3333
xRFM: RMSE = 4332.512043
xRFM: R2   = 0.879093
XGBoost: RMSE = 4248.007124
XGBoost: R2   = 0.883764
Polynomial Regression: RMSE = 4551.132385
Polynomial Regression: R2   = 0.866583
Optimized Ensemble: RMSE = 4219.848306
Optimized Ensemble: R2   = 0.885299


In [22]:
print("Success:", result.success)
print("Message:", result.message)
print("Weights:", [f"{w:.8f}" for w in result.x])
print("OOF MSE:", result.fun)

# Bandingkan dengan equal averaging
equal_weights = np.ones(3) / 3

print(
    "Equal-weight OOF RMSE:",
    np.sqrt(mean_squared_error(
        y_train_np,
        oof_predictions @ equal_weights
    ))
)

print(
    "Optimized-weight OOF RMSE:",
    np.sqrt(mean_squared_error(
        y_train_np,
        oof_predictions @ result.x
    ))
)

Success: True
Message: Optimization terminated successfully
Weights: ['0.33333335', '0.33333333', '0.33333335']
OOF MSE: 21536143.645240948
Equal-weight OOF RMSE: 4640.705082908879
Optimized-weight OOF RMSE: 4640.705080614469


In [23]:
import numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error, r2_score

# 1. Target
y_train_np = np.asarray(y_train.to_numpy()).ravel()
y_test_np = np.asarray(y_test.to_numpy()).ravel()

# 2. Train meta-model menggunakan OOF predictions
ridge_meta = RidgeCV(
    alphas=np.logspace(-4, 4, 30),
    cv=5,
    scoring="neg_mean_squared_error",
)

ridge_meta.fit(oof_predictions, y_train_np)

print("Best alpha:", ridge_meta.alpha_)
print("Intercept:", ridge_meta.intercept_)
print("Meta-weights:", ridge_meta.coef_)

# 3. Stack prediksi test dari base models
test_predictions = np.column_stack([
    pred_xrfm,
    pred_xgb,
    pred_poly,
])

pred_stacking = ridge_meta.predict(test_predictions)

# 4. Compare dengan base models dan equal averaging
pred_equal = test_predictions.mean(axis=1)

for name, pred in [
    ("xRFM", pred_xrfm),
    ("XGBoost", pred_xgb),
    ("Polynomial Regression", pred_poly),
    ("Equal Averaging", pred_equal),
    ("Ridge Stacking", pred_stacking),
]:
    rmse = np.sqrt(mean_squared_error(y_test_np, pred))
    r2 = r2_score(y_test_np, pred)

    print(f"\n{name}:")
    print(f"RMSE = {rmse:.6f}")
    print(f"R2   = {r2:.6f}")

Best alpha: 10000.0
Intercept: -81.72169219910211
Meta-weights: [0.19735776 0.64965342 0.15932285]

xRFM:
RMSE = 4332.512043
R2   = 0.879093

XGBoost:
RMSE = 4248.007124
R2   = 0.883764

Polynomial Regression:
RMSE = 4551.132385
R2   = 0.866583

Equal Averaging:
RMSE = 4219.848299
R2   = 0.885299

Ridge Stacking:
RMSE = 4183.132158
R2   = 0.887287


In [24]:
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor


In [25]:
from sklearn.metrics import mean_squared_error, r2_score
y_pred_test = best_model.predict(X_test_preprocessed).reshape(-1)
print("Mean Squared Error (MSE):", mean_squared_error(y_test, y_pred_test))
print("R-squared (R2):", r2_score(y_test, y_pred_test))

Mean Squared Error (MSE): 17319874.077024195
R-squared (R2): 0.8884379027731897
